In [ ]:
import requests
import pandas as pd
import apimoex
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import datetime as dt
import os
import sklearn
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

In [ ]:
path = r'data/OUT'
data = pd.read_csv(path+'/Pos+quotes+volume.csv')
interest_population = list(data['isin'].unique())

In [ ]:
data.head(2)

In [ ]:
#выбор актива
def Graphs(path,ISIN,data):
    try:    
        df_out = data[data['isin'] == ISIN]
        
        #График
        x_titles = np.array(df_out['moment'].to_list())
        y_values = np.array(df_out['close'].to_list())
        
        fig,ax = plt.subplots()
        
        plt.ylabel('Стоимость контракта, руб.')
        
        ax.plot(x_titles , y_values , color='b',label=df_out[['name']].iloc[0][0])
        plt.xticks(x_titles[::10],size = 8,rotation = 45)
        plt.legend(loc = 2)
        
        
        approximator = df_out['long_position_jur']+df_out['long_position_fiz']
        
        
        x_titles1 = np.array(df_out['moment'].to_list())
        y_values1 = np.array(approximator.to_list())
        
        ax2 = ax.twinx ()
        ax2.plot(x_titles1, y_values1[:] , color='r', label='Отрытые позиции')
        
        # x_titles3 = np.array(df_out['moment'].to_list())
        # y_values3 = np.array(df_out['volume'].to_list())
        
        # ax3.hist(y_values3[:] , color='g')
        
        plt.legend(loc = 4)
        plt.ylabel("Кол-во контрактов")
        
        fig.set_size_inches(10, 6)
        # plt.show()
        plt.savefig(path+ISIN+'_1.png')
        plt.show()
    
        
        #Регрессии
        offset = 60
        df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
        best_appr = pd.DataFrame()
        for i in range(offset):
            Y = np.array(df_out['close'].to_list()).reshape(-1,1)
            X = np.array(approximator.to_list()).reshape(-1,1)
            
            Y = Y[i:]
            X = X[:len(X)-i]
            reg = LinearRegression().fit(X, Y)
            # print('Offset: ' , i)
            # print('R^2 = ', reg.score(X,Y))
            # print('intercept:', reg.intercept_)
            # print('slope:', reg.coef_)
            df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out[['name']].iloc[0][0]] 
        best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
        display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])
        
        plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
        plt.ylim(0,1)
        plt.xlabel(ISIN)
        # plt.show()
        plt.savefig(path+ISIN+'_2.png')
        
        
        #График со сдвигом
        Y = np.array(df_out['close'].to_list()).reshape(-1,1)
        X = np.array(approximator.to_list()).reshape(-1,1)
        
        i = df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]][['offset']].iloc[0].iloc[0]
        
        Y = Y[i:]
        X = X[:len(X)-i]
        
        x_titles = np.array(df_out['moment'].to_list())[i:]
        y_values = Y
        
        fig,ax = plt.subplots()
        
        plt.ylabel('Стоимость контракта, руб.')
        
        ax.plot(x_titles , y_values , color='b',label=df_out[['name']].iloc[0][0])
        plt.xticks(x_titles[::10],size = 8,rotation = 45)
        plt.legend(loc = 2)
        
        
        approximator = df_out['long_position_jur']+df_out['long_position_fiz']
        
        
        y_values1 = X
        
        ax2 = ax.twinx ()
        ax2.plot(x_titles, y_values1[:] , color='r', label='Отрытые позиции')
        
        plt.legend(loc = 4)
        plt.ylabel("Кол-во контрактов")
        
        fig.set_size_inches(10, 6)
        # plt.show()
        plt.savefig(path+ISIN+'_3.png')
    except:None

In [ ]:
Graphs(r'plots/triple_Graphs/','MIX',pd.read_csv(r'data/OUT/Pos+quotes+volume.csv'))

In [ ]:
df_out.info()

In [ ]:
data = pd.read_csv(path+'/Pos+quotes+volume.csv')
ISIN = 'MIX'
df_out = data[data['isin']==ISIN]


offset = 60
df_OLS_stat = pd.DataFrame(columns=['offset','R^2','intercept','slope','isin'])
best_appr = pd.DataFrame()
for i in range(offset):
    Y = np.array(df_out['close'].to_list()).reshape(-1,1)
    X1 = np.array(df_out['long_position'].to_list()).reshape(-1,1)
    X2 = np.array(df_out['long_position_jur'].to_list()).reshape(-1,1)
    X3 = np.array(df_out['long_position_fiz'].to_list()).reshape(-1,1)

    
    Y = Y[i:]
    X1 = X1[:len(X1)-i]
    X2 = X2[:len(X2)-i]
    X3 = X3[:len(X3)-i]
    X = np.column_stack((X1,X2,X3))
    reg = LinearRegression().fit(X, Y)
    # print('Offset: ' , i)
    # print('R^2 = ', reg.score(X,Y))
    # print('intercept:', reg.intercept_)
    # print('slope:', reg.coef_)
    df_OLS_stat.loc[len(df_OLS_stat.index)] = [i, reg.score(X,Y), reg.intercept_[0],reg.coef_[0][0],df_out[['name']].iloc[0][0]] 
best_appr = pd.concat([best_appr, df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]]], ignore_index=True)
display(df_OLS_stat.iloc[[df_OLS_stat['R^2'].idxmax()]])

plt.bar(df_OLS_stat['offset'],df_OLS_stat['R^2'])
plt.ylim(0,1)
plt.xlabel(ISIN)
plt.show()

In [ ]:
df_OLS_stat

In [ ]:
print('Коэффициенты:', reg.coef_)
print('Пересечение:', reg.intercept_)